# 01e — 第一讲基础习题参考答案（Solutions）

本文件对应 `01e_exercises.ipynb` 中全部 10 道基础巩固习题的完整参考代码与解析。

---

## 第一部分：Python 基础语法（对应 `01a`）

### 习题 1：变量与算术运算（带隙与吸收波长换算）

In [ ]:
bandgap_ev = 1.55
wavelength_nm = 1240 / bandgap_ev
print(f"吸收截止波长: {wavelength_nm:.1f} nm")

### 习题 2：列表索引、切片与内置统计函数

In [ ]:
pce_list = [16.2, 16.8, 15.9, 17.1, 16.5]

first_three = pce_list[:3]
max_pce = max(pce_list)
mean_pce = sum(pce_list) / len(pce_list)

print("前 3 个器件效率:", first_three)
print("最高效率:", max_pce)
print("平均效率:", round(mean_pce, 2))

### 习题 3：字典操作与列表推导式

In [ ]:
mol_weights = {
    "benzene": 78.11,
    "naphthalene": 128.17,
    "anthracene": 178.23,
    "pentacene": 278.35,
}

mol_weights["pyrene"] = 202.25
heavy_mols = [name for name, mw in mol_weights.items() if mw > 150]
print("分子量 > 150 的分子:", heavy_mols)

### 习题 4：编写函数与条件分支（`if / elif / else`）

In [ ]:
def classify_solubility(logS):
    if logS >= -2.0:
        return "Soluble"
    elif logS >= -4.0:
        return "Slightly Soluble"
    else:
        return "Insoluble"

test_values = [-0.8, -3.1, -5.6]
print([classify_solubility(v) for v in test_values])

---

## 第二部分：NumPy 与 Pandas 数据处理（对应 `01b`）

### 习题 5：NumPy 数组向量化运算

In [ ]:
import numpy as np

conc_mM = np.array([0.5, 1.2, 2.5, 5.0, 10.0])
conc_M = conc_mM / 1000.0
logS = np.log10(conc_M).round(2)
print("换算后的 logS:", logS)

### 习题 6：Pandas 多条件筛选与排序

In [ ]:
import pandas as pd
from pathlib import Path
from rdkit import Chem
from rdkit.Chem import Descriptors

for p in [Path("../data/esol.csv"), Path("data/esol.csv"), Path("../../01 - Basics/data/esol.csv")]:
    if p.exists():
        df = pd.read_csv(p)
        break

df = df.rename(columns={
    "Compound ID": "name",
    "measured log solubility in mols per litre": "logS",
})[["name", "smiles", "logS"]].copy()
mols = df["smiles"].apply(Chem.MolFromSmiles)
df["MolWt"] = [Descriptors.MolWt(m) for m in mols]
df["MolLogP"] = [Descriptors.MolLogP(m) for m in mols]
df["NumAromaticRings"] = [Descriptors.NumAromaticRings(m) for m in mols]

sub_df = df[(df["NumAromaticRings"] == 2) & (df["logS"] > -2.0)].sort_values("logS", ascending=False)
print(f"满足条件的分子总数: {len(sub_df)}")
sub_df.head(5)

### 习题 7：Pandas 分组聚合（`groupby`）

In [ ]:
ring_summary = df.groupby("NumAromaticRings")["logS"].agg(["count", "mean"]).round(2)
display(ring_summary)

**结果解析**：随着分子中芳香环数量从 `0` 增加到 `5` 以上，平均水溶解度 `logS` 从 `-2.27` 一路单调下降到 `-6.5` 以下。因为稠合或串联的芳香环既增加了疏水性（$\log P$ 升高），又通过 $\pi\text{-}\pi$ 堆积显著提高了晶体晶格能。

---

## 第三部分：科研绘图与 RDKit 分子操作（对应 `01c` & `01d`）

### 习题 8：使用 Matplotlib 绘制并排子图（直方图 + 散点图）

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].hist(df["MolWt"], bins=25, color="steelblue", edgecolor="white")
axes[0].set_xlabel("Molecular Weight (g/mol)")
axes[0].set_ylabel("Count")
axes[0].set_title("Distribution of Molecular Weight")

axes[1].scatter(df["MolWt"], df["logS"], s=10, alpha=0.5, color="darkorange")
axes[1].set_xlabel("Molecular Weight (g/mol)")
axes[1].set_ylabel("Measured logS (mol/L)")
axes[1].set_title("logS vs Molecular Weight")

plt.tight_layout()
plt.show()

<!-- ai4m-style -->
> **📘 课程统一绘图风格**：下面这格加载课程自带的 `ai4m.mplstyle`（配色与课程网站同源，无上右边框，仅横向浅网格，保存默认 300 dpi）。`C` 是课程配色，手动指定颜色时写 `color=C.crimson`。


In [ ]:
# ai4m-style：课程统一绘图风格（本地与 Colab 通用）
import pathlib, sys

sys.path.insert(0, str(pathlib.Path.cwd().parent))   # course/ 根目录
import ai4m_style as ai4m

C = ai4m.use()        # 应用 ai4m.mplstyle，返回课程配色


### 习题 9：RDKit 分子解析与物理化学描述符计算

In [ ]:
para_smiles = "CC(=O)Nc1ccc(O)cc1"
mol_para = Chem.MolFromSmiles(para_smiles)

print(f"MolWt         : {Descriptors.MolWt(mol_para):.2f}")
print(f"MolLogP       : {Descriptors.MolLogP(mol_para):.2f}")
print(f"NumHDonors    : {Descriptors.NumHDonors(mol_para)}")
print(f"NumHAcceptors : {Descriptors.NumHAcceptors(mol_para)}")
mol_para

### 习题 10：Morgan 分子指纹与 Tanimoto 相似度计算

In [ ]:
from rdkit import DataStructs
from rdkit.Chem import rdFingerprintGenerator

mol_phenol = Chem.MolFromSmiles("Oc1ccccc1")
mol_cresol = Chem.MolFromSmiles("Cc1ccc(O)cc1")
mol_cyclo  = Chem.MolFromSmiles("C1CCCCC1")

fp_gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)

fp_phenol = fp_gen.GetFingerprint(mol_phenol)
fp_cresol = fp_gen.GetFingerprint(mol_cresol)
fp_cyclo  = fp_gen.GetFingerprint(mol_cyclo)

sim_1 = DataStructs.TanimotoSimilarity(fp_phenol, fp_cresol)
sim_2 = DataStructs.TanimotoSimilarity(fp_phenol, fp_cyclo)

print(f"苯酚 vs 对甲苯酚 Tanimoto 相似度: {sim_1:.3f}")
print(f"苯酚 vs 环己烷   Tanimoto 相似度: {sim_2:.3f}")